# Transform billers Data

1. Read bronze `billers` table
1. Drop unwanted columns
1. Standardise column names using snake_case 
1. Filter out rows where `txn_id` is null (business key validation)
1. Remove duplicate records
1. Write the transformed data to silver `circuits` table


In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
import pyspark.sql.functions as F

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.billers"
silver_table = f"{catalog_name}.{silver_schema}.billers"

In [0]:
billers_df = spark.read.table(bronze_table).filter(F.col("batch_id")==v_batch_id)

In [0]:
billers_renamed = billers_df.withColumnsRenamed({"BillerID":"biller_id","BillerName":"biller_name","BillerCategory":"biller_category","Status":"status"})

In [0]:
billers_final = billers_renamed.filter(F.col("biller_ID").isNotNull()).dropDuplicates()

In [0]:
billers_final = billers_final.withColumn("created_timestamp", F.current_timestamp()) \
                    .withColumn("updated_timestamp", F.current_timestamp())

In [0]:
%sql
DROP TABLE IF EXISTS payment_app.silver.settlement

In [0]:

if not spark.catalog.tableExists(silver_table):
    (
        billers_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:

    billers_final.createOrReplaceTempView("vw_billers_final")

    spark.sql(f"""
        MERGE INTO {silver_table} AS tgt
        USING vw_billers_final AS src
        ON tgt.biller_id = src.biller_id

        WHEN MATCHED
        AND src.batch_id >= tgt.batch_id
        THEN UPDATE SET
            tgt.biller_name         = src.biller_name,
            tgt.biller_category     = src.biller_category,
            tgt.status              = src.status,
            tgt.batch_id            = src.batch_id,
            tgt.ingestion_timestamp = src.ingestion_timestamp,
            tgt.source_file         = src.source_file,
            tgt.updated_timestamp   = current_timestamp()

        WHEN NOT MATCHED
        THEN INSERT (
            biller_id,
            biller_name,
            biller_category,
            status,
            batch_id,
            ingestion_timestamp,
            source_file,
            created_timestamp,
            updated_timestamp
        )
        VALUES (
            src.biller_id,
            src.biller_name,
            src.biller_category,
            src.status,
            src.batch_id,
            src.ingestion_timestamp,
            src.source_file,
            src.created_timestamp,
            src.updated_timestamp
        )
    """)

In [0]:
display(spark.table(silver_table))